In [1]:
import pandas as pd

lap_times = pd.read_csv('AdvanceData/data/lap_times.csv')
results = pd.read_csv('AdvanceData/data/results.csv')
races = pd.read_csv('AdvanceData/data/races.csv')
constructors = pd.read_csv('AdvanceData/data/constructors.csv')
constructor_standings = pd.read_csv('AdvanceData/data/constructor_standings.csv')

lap_times.head()


,raceId,driverId,lap,position,time,milliseconds
0,841,20,1,1,1:38.109,98109
1,841,20,2,1,1:33.006,93006
2,841,20,3,1,1:32.713,92713
3,841,20,4,1,1:32.803,92803
4,841,20,5,1,1:32.342,92342


In [2]:
# attach constructorId to each lap (lap_times only has driverId)
laps = lap_times.merge(results[['raceId', 'driverId', 'constructorId']], on=['raceId', 'driverId'])
laps = laps.merge(races[['raceId', 'year']], on='raceId')

# keep 2018-2024, drop lap 1 (often distorted by the start)
laps = laps[(laps['year'] >= 2018) & (laps['year'] <= 2024) & (laps['lap'] > 1)]

laps['seconds'] = laps['milliseconds'] / 1000
laps.head()

,raceId,driverId,lap,position,time,milliseconds,constructorId,year,seconds
426633,989,843,57,15,1:28.176,88176,5,2018,88.176
426634,989,843,56,15,1:28.493,88493,5,2018,88.493
426635,989,843,55,15,1:28.326,88326,5,2018,88.326
426636,989,843,54,15,1:28.255,88255,5,2018,88.255
426637,989,843,53,15,1:28.298,88298,5,2018,88.298


In [4]:
# median lap time per constructor per race (median resists outlier laps: pit stops, safety car)
team_pace = laps.groupby(['year', 'raceId', 'constructorId'])['seconds'].median().reset_index()
team_pace.columns = ['year', 'raceId', 'constructorId', 'median_lap_time']

# fastest team pace in each race = reference
fastest_per_race = team_pace.groupby('raceId')['median_lap_time'].transform('min')
team_pace['pace_gap'] = team_pace['median_lap_time'] - fastest_per_race

team_pace.head()
pre_cap_standings = constructor_standings.merge(races[['raceId', 'year', 'round']], on='raceId')
idx = pre_cap_standings.groupby(['year', 'constructorId'])['round'].idxmax()
pre_cap_final = pre_cap_standings.loc[idx]
pre_cap_final = pre_cap_final[pre_cap_final['year'].between(2018, 2020)]

avg_points_pre_cap = pre_cap_final.groupby('constructorId')['points'].mean().sort_values(ascending=False)
big_three_ids = avg_points_pre_cap.head(3).index.tolist()

big_three_names = constructors[constructors['constructorId'].isin(big_three_ids)]['name'].tolist()
print("Big three (avg points 2018-2020):", big_three_names)

Big three (avg points 2018-2020): ['Ferrari', 'Red Bull', 'Mercedes']
